# Step 1: Prepare the English MINDS-14 dataset

Load the `en-US`, `en-GB`, and `en-AU` configurations from `PolyAI/minds14`. This notebook prepares the external evaluation data only; it does not alter the synthetic dataset, train a model, or score predictions.

## 1. Set up the Python environment and project paths

Use the project Python 3.12 kernel, set the shared seed, import the Hugging Face dataset loader and pandas, and add the repository root so paths and project files resolve from this notebook.

In [1]:
import random
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from datasets import load_dataset

sys.path.insert(0, '..')
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
DATASET_NAME = 'PolyAI/minds14'
LOCALES = ['en-US', 'en-GB', 'en-AU']
OUTPUT_PATH = Path('../data/processed/minds14_en.csv')
print(f'Python: {sys.version.split()[0]}')
print(f'Output: {OUTPUT_PATH}')

Python: 3.12.3
Output: ..\data\processed\minds14_en.csv


## 2. Load the three English locale configurations

For each configuration, retain only the transcription and intent-class fields. Convert the integer class with the dataset feature's `int2str` mapping and attach the locale identifier. Audio is never included in the prepared records.

In [2]:
records = []
locale_row_counts = {}
locale_intent_names = {}

for locale in LOCALES:
    ds = load_dataset(DATASET_NAME, locale, split='train')
    intent_feature = ds.features['intent_class']
    locale_intent_names[locale] = set(intent_feature.names)
    selected_ds = ds.select_columns(['transcription', 'intent_class'])
    locale_count = 0

    for row in selected_ds:
        records.append(
            {
                'text': row['transcription'],
                'intent': ds.features['intent_class'].int2str(row['intent_class']),
                'locale': locale,
            }
        )
        locale_count += 1

    locale_row_counts[locale] = locale_count

minds14_df = pd.DataFrame.from_records(records, columns=['text', 'intent', 'locale'])
print('Rows loaded by locale:', locale_row_counts)
print('Combined shape:', minds14_df.shape)
minds14_df.head()

README.md:   0%|          | 0.00/23.5k [00:00<?, ?B/s]

c:\Users\mahmu\miniconda3\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\mahmu\.cache\huggingface\hub\datasets--PolyAI--minds14. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\mahmu\miniconda3\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system 

en-US/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 34.2MB            

en-US/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/563 [00:00<?, ? examples/s]

...

en-AU/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 37.3MB            

en-AU/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/654 [00:00<?, ? examples/s]

Rows loaded by locale: {'en-US': 563, 'en-GB': 592, 'en-AU': 654}
Combined shape: (1809, 3)


,text,intent,locale
0,I would like to set up a joint account with my...,joint_account,en-US
1,Henry County set up a joint account with my wi...,joint_account,en-US
2,hi I'd like to set up a joint account with my ...,joint_account,en-US
3,how do I start a joint account,joint_account,en-US
4,can you help me set up a joint bank account,joint_account,en-US


## 3. Verify schema and intent-label consistency

Compare the MINDS-14 intent names with the 14 labels in the existing synthetic training split. This is a verification only: do not rename or force-map external labels. A mismatch is reported explicitly so the team can resolve it before evaluating classifiers.

In [3]:
expected_intents = set(
    pd.read_csv('../data/processed/train.csv', usecols=['intent'])['intent'].dropna().unique()
)
observed_intents = set(minds14_df['intent'].dropna().unique())
missing_from_minds14 = sorted(expected_intents - observed_intents)
additional_in_minds14 = sorted(observed_intents - expected_intents)
labels_match_exactly = expected_intents == observed_intents

assert minds14_df.columns.tolist() == ['text', 'intent', 'locale']
assert len(expected_intents) == 14, f'Expected 14 project intents, found {len(expected_intents)}'
assert minds14_df['text'].notna().all(), 'MINDS-14 contains missing transcriptions'
assert minds14_df['intent'].notna().all(), 'MINDS-14 contains missing intent labels'
assert minds14_df['locale'].isin(LOCALES).all(), 'Unexpected locale value found'

print('Expected project intent count:', len(expected_intents))
print('Observed MINDS-14 intent count:', len(observed_intents))
print('Intent names match exactly:', labels_match_exactly)
print('Project intents missing from MINDS-14:', missing_from_minds14)
print('Additional MINDS-14 intents:', additional_in_minds14)
print('Intent names by locale:', {locale: sorted(names) for locale, names in locale_intent_names.items()})

Expected project intent count: 14
Observed MINDS-14 intent count: 14
Intent names match exactly: True
Project intents missing from MINDS-14: []
Additional MINDS-14 intents: []
Intent names by locale: {'en-US': ['abroad', 'address', 'app_error', 'atm_limit', 'balance', 'business_loan', 'card_issues', 'cash_deposit', 'direct_debit', 'freeze', 'high_value_payment', 'joint_account', 'latest_transactions', 'pay_bill'], 'en-GB': ['abroad', 'address', 'app_error', 'atm_limit', 'balance', 'business_loan', 'card_issues', 'cash_deposit', 'direct_debit', 'freeze', 'high_value_payment', 'joint_account', 'latest_transactions', 'pay_bill'], 'en-AU': ['abroad', 'address', 'app_error', 'atm_limit', 'balance', 'business_loan', 'card_issues', 'cash_deposit', 'direct_debit', 'freeze', 'high_value_payment', 'joint_account', 'latest_transactions', 'pay_bill']}


## 4. Save and inspect the prepared data

Write only the requested three-column CSV to `data/processed/minds14_en.csv`. Then reload it from disk to verify its schema and report counts by intent, locale, and their cross-tabulation.

In [4]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
minds14_df.to_csv(OUTPUT_PATH, index=False)
verified_df = pd.read_csv(OUTPUT_PATH)

assert verified_df.columns.tolist() == ['text', 'intent', 'locale']
assert verified_df['text'].notna().all()
assert verified_df['intent'].notna().all()
assert verified_df['locale'].isin(LOCALES).all()

print(f'Saved: {OUTPUT_PATH.resolve()}')
print(f'Rows: {len(verified_df)}')
print('Intent distribution:')
display(verified_df['intent'].value_counts().sort_index().rename('rows').to_frame())
print('Locale distribution:')
display(verified_df['locale'].value_counts().sort_index().rename('rows').to_frame())
print('Intent by locale:')
display(pd.crosstab(verified_df['intent'], verified_df['locale'], margins=True))

Saved: C:\Users\mahmu\Intent_Classification_For_Banking\data\processed\minds14_en.csv
Rows: 1809
Intent distribution:


,rows
intent,
abroad,115
address,131
app_error,125
atm_limit,137
balance,131
business_loan,130
card_issues,138
cash_deposit,135
direct_debit,125


Locale distribution:


,rows
locale,
en-AU,654
en-GB,592
en-US,563


Intent by locale:


locale,en-AU,en-GB,en-US,All
intent,,,,
abroad,35,46,34,115
address,52,45,34,131
app_error,42,41,42,125
atm_limit,50,46,41,137
balance,48,42,41,131
business_loan,46,45,39,130
card_issues,48,44,46,138
cash_deposit,43,44,48,135
direct_debit,47,42,36,125
